[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/tracing/llama_index_openai_agent_tracing_tutorial.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">追踪与评估 LlamaIndex OpenAI 智能体应用</h1>

借助支持函数调用的 OpenAI API，构建自己的智能体变得更加容易。

本 Notebook 教程展示如何用不到 50 行代码编写 OpenAI 智能体，并使用 Phoenix 检查其内部执行过程。实现虽精简，但具备完整功能，包括持续对话和使用工具。

安装 LlamaIndex 及其他依赖。

In [ ]:
!pip install "openai>=1" "arize-phoenix>=4.29.0" "llama_index>=0.11.0"

导入库。

In [ ]:
import os
from getpass import getpass

import openai
import pandas as pd
from llama_index.agent.openai import OpenAIAgent
from llama_index.core.prompts.system import SHAKESPEARE_WRITING_ASSISTANT
from llama_index.core.tools import FunctionTool
from llama_index.llms.openai import OpenAI

import phoenix as px

pd.set_option("display.max_colwidth", 1000)

可以在后台运行 Phoenix，收集已通过 `OpenInferenceTraceCallbackHandler` 添加埋点的 LlamaIndex 应用发出的追踪数据。

启动 Phoenix，按照单元输出中的说明打开 Phoenix 界面。由于尚未运行 LlamaIndex 应用，此时界面应为空。

In [ ]:
(session := px.launch_app()).view()

通过 `LlamaIndexInstrumentor` 启用 Phoenix 追踪。Phoenix 使用 OpenInference 追踪：这是一种用于捕获和存储 LLM 应用追踪的开源标准，使应用能够无缝接入 Phoenix 等 LLM 可观测性解决方案。

In [ ]:
from openinference.instrumentation.llama_index import LlamaIndexInstrumentor
from phoenix.otel import register

tracer_provider = register(endpoint="http://127.0.0.1:6006/v1/traces")
LlamaIndexInstrumentor().instrument(skip_dep_check=True, tracer_provider=tracer_provider)

首先导入一些基础组件，并定义智能体可以使用的工具。

In [ ]:
def multiply(a: int, b: int) -> int:
    """将两个整数相乘并返回整数结果"""
    return a * b


multiply_tool = FunctionTool.from_defaults(fn=multiply)


def add(a: int, b: int) -> int:
    """将两个整数相加并返回整数结果"""
    return a + b


add_tool = FunctionTool.from_defaults(fn=add)

提供 API 密钥以访问 OpenAI。

In [ ]:
if not (openai_api_key := os.getenv("OPENAI_API_KEY")):
    openai_api_key = getpass("🔑 请输入 OpenAI API 密钥： ")
openai.api_key = openai_api_key
os.environ["OPENAI_API_KEY"] = openai_api_key

现在定义一个能够对话并调用工具的智能体。

核心逻辑位于 chat 方法中，大致分为三步：

- 调用 OpenAI，决定是否调用工具、调用哪个工具，以及使用什么参数。
- 使用这些参数调用工具，获得输出。
- 调用 OpenAI，根据对话上下文和工具输出生成回答。

reset 方法会重置对话上下文，便于开始新的对话。

为了增加趣味，让智能体使用莎士比亚的风格对话。

In [ ]:
llm = OpenAI(model="gpt-4o")
agent = OpenAIAgent.from_tools(
    [multiply_tool, add_tool],
    llm=llm,
    system_prompt=SHAKESPEARE_WRITING_ASSISTANT,
)

现在来和智能体聊天！

In [ ]:
response = agent.query("What is (121 * 3) + 42?")
print(response)

再聊几轮，这次提出一些追问。

In [ ]:
queries = [
    "What is (121 * 3) + 42?",
    "what is 3 * 3?",
    "what is 4 * 4?",
    "what is 75 * (3 + 4)?",
    "what is 23 times 87",
]

for query in queries:
    print(f"> {query}")
    response = agent.query(query)
    print(response)
    agent.reset()
    print("---")

在浏览器中打开 `session.url`，查看 Phoenix 中的追踪。注意，LLM span 包含 OpenAI 函数调用，可以据此查看 LLM 根据查询选择了哪个工具。

有关函数调用的更多信息，请参阅 [OpenAI API 文档](https://openai.com/blog/function-calling-and-other-api-updates)。


In [ ]:
print(f"若尚未打开，请打开 Phoenix 界面： {session.url}")

还可以将智能体的聊天记录作为 dataframe 查看。

In [ ]:
from phoenix.client import Client

df = Client().spans.get_spans_dataframe()
df.head()